In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import gensim
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate, BatchNormalization, Reshape, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import torch
from transformers import AutoTokenizer, AutoModel

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
kaggle_df = pd.read_csv('/content/drive/MyDrive/thesis/kaggle/part2 feature-creation/2.3 feature engineering/kaggle_df.csv')

kaggle_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Username                     2000 non-null   object 
 1   Display_Name                 2000 non-null   object 
 2   Gender                       2000 non-null   object 
 3   notebook_url                 2000 non-null   object 
 4   code_location                2000 non-null   object 
 5   labels                       2000 non-null   object 
 6   top_labels                   2000 non-null   object 
 7   code_sections                2000 non-null   object 
 8   markdown_sections            2000 non-null   object 
 9   all_sections                 2000 non-null   object 
 10  only_code_in_code_sections   2000 non-null   object 
 11  number_of_lines              2000 non-null   float64
 12  names_set                    2000 non-null   object 
 13  num_of_sections   

In [4]:
kaggle_df.head()

,Username,Display_Name,Gender,notebook_url,code_location,labels,top_labels,code_sections,markdown_sections,all_sections,...,function_density,loop_density,condition_density,comment_tokens_density,avg_var_name_length,comment_to_code_ratio,avg_func_length,code_to_markdown_ratio,avg_markdown_lines_length,markdown_sentiment
0,tchaye59,Jude TCHAYE,male,https://www.kaggle.com/code/tchaye59/jmarket-k...,/content/drive/MyDrive/thesis/notebooks/male/t...,"['Jane Street Market Prediction', 'Jane Street...",{'Jane Street Market Prediction'},['# This Python 3 environment comes with many ...,['### This notebook is only dedicated to submi...,['# This Python 3 environment comes with many ...,...,0.000000,0.019355,0.000000,0.524272,6.333333,0.125708,0.000000,7.675676,1.000000,0.152933
1,iyara1,Riya,female,https://www.kaggle.com/code/iyara1/deepanalysi...,/content/drive/MyDrive/thesis/notebooks/female...,['House Prices - Advanced Regression Techniques'],{'House Prices - Advanced Regression Techniques'},"[""import numpy as np\nimport pandas as pd\nimp...",['**#Bivariate Analysis******'],"[""import numpy as np\nimport pandas as pd\nimp...",...,0.002674,0.018717,0.018717,0.359568,6.166667,0.002423,4.000000,398.903226,1.000000,0.000000
2,sanjay7013,Sanjay M,male,https://www.kaggle.com/code/sanjay7013/credit-...,/content/drive/MyDrive/thesis/notebooks/male/s...,['Credit Card Fraud Detection'],{'Credit Card Fraud Detection'},['# This Python 3 environment comes with many ...,"['# Credit Card Fraud Detection', ""### DataSet...","['# Credit Card Fraud Detection', ""### DataSet...",...,0.000000,0.009709,0.000000,0.386076,6.846154,0.542884,0.000000,1.788180,1.235294,0.004424
3,validmodel,Rashmi Margani,female,https://www.kaggle.com/code/validmodel/master-...,/content/drive/MyDrive/thesis/notebooks/female...,['Iris Species'],{'Iris Species'},['# This Python 3 environment comes with many ...,"[""# <h1 style='background:#f0c2c1; border:2; b...",['# This Python 3 environment comes with many ...,...,0.039735,0.029801,0.039735,0.279314,7.084211,0.755798,6.333333,1.261214,8.736842,0.414937
4,rajeevnair676,Rajeev Nair,male,https://www.kaggle.com/code/rajeevnair676/nlp-...,/content/drive/MyDrive/thesis/notebooks/male/r...,"['IMDB Dataset of 50K Movie Reviews', 'Tweet S...",{'Natural Language Processing with Disaster Tw...,['#Importing NLTK package\nimport nltk\n\nimpo...,['# <center><b> NLP STARTERS - PART 1 <center/...,['# <center><b> NLP STARTERS - PART 1 <center/...,...,0.025381,0.086294,0.015228,0.193370,6.698413,2.530189,11.400000,0.377135,5.000000,0.407079


In [5]:
X=kaggle_df.drop('Gender',axis=1)
Y=kaggle_df.Gender.map({"male": 1, "female": 0})

In [6]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 29 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Username                     2000 non-null   object 
 1   Display_Name                 2000 non-null   object 
 2   notebook_url                 2000 non-null   object 
 3   code_location                2000 non-null   object 
 4   labels                       2000 non-null   object 
 5   top_labels                   2000 non-null   object 
 6   code_sections                2000 non-null   object 
 7   markdown_sections            2000 non-null   object 
 8   all_sections                 2000 non-null   object 
 9   only_code_in_code_sections   2000 non-null   object 
 10  number_of_lines              2000 non-null   float64
 11  names_set                    2000 non-null   object 
 12  num_of_sections              2000 non-null   int64  
 13  token_count       

# Non text features

In [7]:
X_nontext=X.select_dtypes(exclude=['object'])
X_nontext.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   number_of_lines              2000 non-null   float64
 1   num_of_sections              2000 non-null   int64  
 2   token_count                  2000 non-null   int64  
 3   variables_count              2000 non-null   int64  
 4   function_count               2000 non-null   int64  
 5   loop_count                   2000 non-null   int64  
 6   condition_count              2000 non-null   int64  
 7   single_line_comment_density  2000 non-null   float64
 8   function_density             2000 non-null   float64
 9   loop_density                 2000 non-null   float64
 10  condition_density            2000 non-null   float64
 11  comment_tokens_density       2000 non-null   float64
 12  avg_var_name_length          2000 non-null   float64
 13  comment_to_code_ra

In [8]:
X_train_nontext, X_test_nontext, y_train, y_test = train_test_split(X_nontext, Y, test_size=0.2, random_state=0,stratify=Y)

In [9]:
# Preprocess non-text features
scaler = StandardScaler()
X_train_nontext = scaler.fit_transform(X_train_nontext)
X_test_nontext = scaler.transform(X_test_nontext)

In [10]:
def concatenate_code_sections(row, unique_char):
    code_list = eval(row)
    concatenated_code = unique_char.join(code_list)
    return concatenated_code

unique_char = ' \x1f '  # Using Unit Separator (ASCII 31) as a unique character

X['parsed_code'] = X['code_sections'].apply(concatenate_code_sections, unique_char=unique_char)

In [11]:
X_train_text, X_test_text, y_train, y_test = train_test_split(X.parsed_code, Y, test_size=0.2, random_state=0,stratify=Y)

## CodeBert

In [12]:
import torch
from transformers import RobertaTokenizer, RobertaModel

def load_codebert_model():
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")
    model = RobertaModel.from_pretrained("microsoft/codebert-base")
    return tokenizer, model

def embed_python_file(code):

    inputs = tokenizer(code, return_tensors="pt", max_length=512, truncation=True, padding="max_length")

    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].numpy().flatten()
    return embedding

tokenizer, model = load_codebert_model()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [13]:
# X_train_codebert = X_train_text.apply(embed_python_file)

X_train_codebert = pd.Series(pd.read_csv('/content/drive/MyDrive/thesis/kaggle/part3 model-enhancment/train_codebert-kaggle.csv', index_col= 0).parsed_code.apply(lambda string_val: np.array([eval(element) for element in string_val.replace('[','').replace(']','').split()])))
X_train_codebert = np.array([np.array(val) for val in X_train_codebert])

In [14]:
# X_test_codebert = X_test_text.apply(embed_python_file)

X_test_codebert = pd.Series(pd.read_csv('/content/drive/MyDrive/thesis/kaggle/part3 model-enhancment/test_codebert-kaggle.csv', index_col= 0).parsed_code.apply(lambda string_val: np.array([eval(element) for element in string_val.replace('[','').replace(']','').split()])))
X_test_codebert = np.array([np.array(val) for val in X_test_codebert])

In [15]:
embedding_dim = X_train_codebert.shape[1]

In [16]:
# text branch
text_input = Input(shape=(embedding_dim,), name='text_input')
dense_text = Dense(32, activation='relu')(text_input)
dense_text = BatchNormalization()(dense_text)
dense_text = Dropout(0.3)(dense_text)

In [17]:
# Non-text branch
non_text_input = Input(shape=(X_train_nontext.shape[1],), name='non_text_input')
dense_non_text = Dense(1, activation='relu')(non_text_input)
dense_non_text = BatchNormalization()(dense_non_text)
dense_non_text = Dropout(0.3)(dense_non_text)

In [18]:
# Concatenate branches
concatenated = Concatenate()([dense_text, dense_non_text])
# dropout = Dropout(0.5)(concatenated)  # Add dropout for regularization
output = Dense(1, activation='sigmoid')(concatenated)

In [19]:
# Create model
model = Model(inputs=[text_input, non_text_input], outputs=output)

# Compile model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 non_text_input (InputLayer  [(None, 18)]                 0         []                            
 )                                                                                                
                                                                                                  
 dense (Dense)               (None, 32)                   24608     ['text_input[0][0]']          
                                                                                                  
 dense_1 (Dense)             (None, 1)                    19        ['non_text_input[0][0]']  

In [20]:
history = model.fit(
    [X_train_codebert, X_train_nontext],
    y_train,
    epochs=100,
    batch_size=64,
    validation_data=([X_test_codebert, X_test_nontext], y_test)
)

Epoch 1/100
25/25 [==============================] - 5s 30ms/step - loss: 0.7199 - accuracy: 0.5275 - val_loss: 0.6864 - val_accuracy: 0.5350
Epoch 2/100
25/25 [==============================] - 0s 8ms/step - loss: 0.6872 - accuracy: 0.5462 - val_loss: 0.7112 - val_accuracy: 0.4825
Epoch 3/100
25/25 [==============================] - 0s 10ms/step - loss: 0.6728 - accuracy: 0.5675 - val_loss: 0.6858 - val_accuracy: 0.5150
Epoch 4/100
25/25 [==============================] - 0s 9ms/step - loss: 0.6646 - accuracy: 0.6000 - val_loss: 0.6894 - val_accuracy: 0.5175
Epoch 5/100
25/25 [==============================] - 0s 8ms/step - loss: 0.6612 - accuracy: 0.6062 - val_loss: 0.6797 - val_accuracy: 0.5350
Epoch 6/100
25/25 [==============================] - 0s 9ms/step - loss: 0.6488 - accuracy: 0.6169 - val_loss: 0.6858 - val_accuracy: 0.5225
Epoch 7/100
25/25 [==============================] - 0s 10ms/step - loss: 0.6388 - accuracy: 0.6306 - val_loss: 0.6821 - val_accuracy: 0.5525
Epoch 8/10

In [21]:
y_pred_test = np.round(model.predict([X_test_codebert, X_test_nontext]))

13/13 [==============================] - 0s 2ms/step


In [22]:
from sklearn.metrics import f1_score
print(f1_score(y_test, y_pred_test))

0.18548387096774194


In [23]:
print(f1_score(y_train, np.round(model.predict([X_train_codebert, X_train_nontext]))))

50/50 [==============================] - 0s 2ms/step
0.3249027237354085


## LSTM

In [24]:
# text branch
text_input = Input(shape=(embedding_dim,), name='text_input')
reshaped = Reshape(target_shape=(1, embedding_dim))(text_input)
lstm_text = LSTM(64, return_sequences=False)(reshaped)

lstm1 = Bidirectional(LSTM(64, return_sequences=True))(reshaped)
lstm1 = BatchNormalization()(lstm1)
lstm1 = Dropout(0.3)(lstm1)

lstm2 = Bidirectional(LSTM(64, return_sequences=True))(lstm1)
lstm2 = BatchNormalization()(lstm2)
lstm2 = Dropout(0.3)(lstm2)

lstm3 = Bidirectional(LSTM(64, return_sequences=False))(lstm2)
lstm3 = BatchNormalization()(lstm3)
lstm3 = Dropout(0.3)(lstm3)

dense_text = Dense(32, activation='relu')(lstm3)
dense_text = BatchNormalization()(dense_text)
dense_text = Dropout(0.3)(dense_text)

In [25]:
# Concatenate branches
concatenated = Concatenate()([dense_text, dense_non_text])
output = Dense(1, activation='sigmoid')(concatenated)

In [26]:
# Create model
model = Model(inputs=[text_input, non_text_input], outputs=output)

# Compile model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 reshape (Reshape)           (None, 1, 768)               0         ['text_input[0][0]']          
                                                                                                  
 bidirectional (Bidirection  (None, 1, 128)               426496    ['reshape[0][0]']             
 al)                                                                                              
                                                                                                  
 batch_normalization_2 (Bat  (None, 1, 128)               512       ['bidirectional[0][0]'] 

In [27]:
# Train the model
history = model.fit(
    [X_train_codebert, X_train_nontext],
    y_train,
    epochs=50,
    batch_size=64,
    validation_data=([X_test_codebert, X_test_nontext], y_test),
)

Epoch 1/50
25/25 [==============================] - 15s 105ms/step - loss: 0.8764 - accuracy: 0.5075 - val_loss: 0.6941 - val_accuracy: 0.5025
Epoch 2/50
25/25 [==============================] - 0s 16ms/step - loss: 0.7833 - accuracy: 0.5431 - val_loss: 0.7005 - val_accuracy: 0.4950
Epoch 3/50
25/25 [==============================] - 0s 16ms/step - loss: 0.7851 - accuracy: 0.5300 - val_loss: 0.7043 - val_accuracy: 0.4850
Epoch 4/50
25/25 [==============================] - 0s 17ms/step - loss: 0.7326 - accuracy: 0.5756 - val_loss: 0.7143 - val_accuracy: 0.4950
Epoch 5/50
25/25 [==============================] - 0s 17ms/step - loss: 0.7252 - accuracy: 0.5544 - val_loss: 0.7124 - val_accuracy: 0.4975
Epoch 6/50
25/25 [==============================] - 0s 17ms/step - loss: 0.7075 - accuracy: 0.5863 - val_loss: 0.7045 - val_accuracy: 0.4875
Epoch 7/50
25/25 [==============================] - 0s 17ms/step - loss: 0.6983 - accuracy: 0.5856 - val_loss: 0.7049 - val_accuracy: 0.4875
Epoch 8/50


## BiLSTM

In [28]:
# text branch
text_input = Input(shape=(embedding_dim,), name='text_input')
reshaped = Reshape((1, embedding_dim))(text_input)
bilstm1 = Bidirectional(LSTM(64, return_sequences=True))(reshaped)
bilstm1 = BatchNormalization()(bilstm1)
bilstm1 = Dropout(0.3)(bilstm1)

bilstm2 = Bidirectional(LSTM(64, return_sequences=False))(bilstm1)
bilstm2 = BatchNormalization()(bilstm2)
bilstm2 = Dropout(0.3)(bilstm2)

# bilstm3 = Bidirectional(LSTM(64, return_sequences=False))(bilstm2)
# bilstm3 = BatchNormalization()(bilstm3)
# bilstm3 = Dropout(0.3)(bilstm3)

dense_text = Dense(4, activation='relu')(bilstm2)
dense_text = BatchNormalization()(dense_text)
dense_text = Dropout(0.3)(dense_text)

In [29]:
# Concatenate branches
concatenated = Concatenate()([dense_text, dense_non_text])
output = Dense(1, activation='sigmoid')(concatenated)

In [30]:
# Create model
model = Model(inputs=[text_input, non_text_input], outputs=output)

# Compile model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 768)]                0         []                            
                                                                                                  
 reshape_1 (Reshape)         (None, 1, 768)               0         ['text_input[0][0]']          
                                                                                                  
 bidirectional_3 (Bidirecti  (None, 1, 128)               426496    ['reshape_1[0][0]']           
 onal)                                                                                            
                                                                                                  
 batch_normalization_6 (Bat  (None, 1, 128)               512       ['bidirectional_3[0][0]'

In [ ]:
# Train the model
history = model.fit(
    [X_train_codebert, X_train_nontext],
    y_train,
    epochs=50,
    batch_size=64,
    validation_data=([X_test_codebert, X_test_nontext], y_test),
)

Epoch 1/50
25/25 [==============================] - 12s 108ms/step - loss: 0.8839 - accuracy: 0.5006 - val_loss: 0.6865 - val_accuracy: 0.5300
Epoch 2/50
25/25 [==============================] - 0s 19ms/step - loss: 0.8022 - accuracy: 0.5288 - val_loss: 0.6885 - val_accuracy: 0.5200
Epoch 3/50
25/25 [==============================] - 1s 20ms/step - loss: 0.7521 - accuracy: 0.5562 - val_loss: 0.6884 - val_accuracy: 0.5150
Epoch 4/50
25/25 [==============================] - 0s 19ms/step - loss: 0.7560 - accuracy: 0.5669 - val_loss: 0.6883 - val_accuracy: 0.5150
Epoch 5/50
25/25 [==============================] - 0s 13ms/step - loss: 0.7235 - accuracy: 0.5938 - val_loss: 0.6869 - val_accuracy: 0.5275
Epoch 6/50
25/25 [==============================] - 0s 14ms/step - loss: 0.7167 - accuracy: 0.5725 - val_loss: 0.6865 - val_accuracy: 0.5250
Epoch 7/50
25/25 [==============================] - 0s 14ms/step - loss: 0.6882 - accuracy: 0.5763 - val_loss: 0.6825 - val_accuracy: 0.5750
Epoch 8/50


In [ ]:
y_pred_train = np.round(model.predict([X_train_codebert, X_train_nontext])).astype(int)

print(classification_report(y_train, y_pred_train))

print("f1_score : ",f1_score(y_train, y_pred_train, average='weighted'))

disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_train, y_pred_train))

disp.plot()

plt.show()

In [ ]:
y_pred_test = np.round(model.predict([X_test_codebert, X_test_nontext])).astype(int)

print(classification_report(y_test, y_pred_test))

print("f1_score : ",f1_score(y_test, y_pred_test, average='weighted'))

disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_test))

disp.plot()

plt.show()